# 5주차 · 어느 점수부터 APS 고장으로 판정할까 — 비용으로 정하는 판정 기준

**⏱ 100분** · Scania APS 고장 기록

## 🎯 이번 주의 질문

스카니아 트럭의 압축 공기 장치(APS) 고장 기록으로, 어느 점수부터 APS 고장으로 판정할지 정합니다. 판정은 두 가지로 틀립니다. 다른 부품 고장을 APS로 판정하는 **오경보**와, APS 고장을 놓치는 **미탐지**입니다. 이 자료를 공개한 챌린지는 오경보 한 건에 10, 미탐지 한 건에 500의 비용을 매깁니다.

> **오경보 비용 10, 미탐지 비용 500일 때 어느 점수부터 APS 고장으로 판정할까요? 흔히 쓰는 0.5보다 비용을 얼마나 줄일까요?**

**이번 주에 할 일**
1. 두 오류를 세는 함수를 만들고, 자료를 학습용·검증용으로 나눈다 (1~3절)
2. 모델로 기록마다 APS일 확률(점수)을 만든다 (4절)
3. 비용으로 기준을 계산하고, 그 기준으로 검증 비용을 센다 (5~6절)
4. 기준을 바꿔 가며 검증 비용이 가장 낮은 기준을 찾는다 (7절)
5. 테스트 자료에서 기본값 0.5와 비교하고 결정한다 (8~9절)

**데이터:** [UCI APS Failure at Scania Trucks](https://archive.ics.uci.edu/dataset/421/aps+failure+at+scania+trucks) — 실제 트럭 운행 기록 중 전문가가 고른 고장 기록. 학습 파일 60,000건, 테스트 파일 16,000건.

**이번 주에 반복할 패턴:** 판정 `(proba_val >= 기준).astype(int)` → `errors(y_true, y_pred)` → 비용 `10 * fp + 500 * fn`

배경지식: **[5주차 배경지식 — 판정과 비용](https://balab-pknu.github.io/bizanalytics/theory/week05.html)**

### 셀 표시 읽는 법

| 표시 | 뜻 | 할 일 |
|---|---|---|
| ▶ | 제공 코드 | 그대로 실행하고 **결과만** 읽습니다 |
| ✍️ | 직접 입력 | 보여 준 코드를 아래 빈 셀에 **손으로 타이핑**합니다. 이번 주의 핵심 패턴입니다 |
| 📝 | 과제 | 앞의 코드를 **조금 바꿔서** 스스로 풉니다 |
| 💬 · ✅ | 결과 해설 · 체크포인트 | 숫자를 읽는 법, 내 결과가 맞는지 확인할 값 |

> **실습본입니다.** 먼저 메뉴 **파일 → Drive에 사본 저장**을 누르세요. 사본을 저장하지 않으면 입력한 코드가 사라집니다.
> 맨 위 준비 셀부터 순서대로 `Shift + Enter`로 실행합니다. ▶ 셀은 그대로 실행하고, ✍️ 셀은 위에 보여 준 코드를
> 빈 셀에 손으로 타이핑하고, 빈칸(`___`)은 채워서 실행합니다. 📝 과제는 힌트를 보고 스스로 풉니다.

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. 학습 파일 train과 테스트 파일 test를 읽습니다.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
import os
import sys

for folder in (".", "..", "../..", "../../.."):
    if os.path.exists(os.path.join(folder, "balab.py")):
        sys.path.insert(0, folder)
        break
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/BALAB-PKNU/bizanalytics/main/balab.py", "balab.py")

from balab import load
aps = load("aps")
train = aps["train"]
test = aps["test"]
print("준비 완료!", train.shape, test.shape)

✅ `준비 완료! (60000, 171) (16000, 171)`이 나오면 됩니다. `test`는 8절의 마지막 평가에만 씁니다.

---
## 1. 자료 보기

학습 파일의 앞부분을 봅니다.

```python
train.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 **한 행 = 트럭 고장 기록 하나**입니다. `class`가 정답이고(`pos` APS 고장, `neg` 다른 부품 고장), 나머지 170개 열은 이름을 가린 수치 기록입니다. 모든 행이 고장 기록이므로, 이 자료는 **APS 고장과 다른 고장을 가려내는 분류**입니다.

APS 고장과 다른 고장의 건수를 셉니다.

```python
train["class"].value_counts()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `neg 59000`, `pos 1000`이면 맞습니다. APS 고장은 60건 중 1건입니다.

모델에 넣을 입력 `X`를 만듭니다. 정답 열 `class`를 뺀 나머지 열입니다.

```python
X = train.drop(columns=["class"])
```

In [ ]:
# ✍️ 위 코드를 입력하세요


`X`의 크기를 확인합니다.

```python
X.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 60,000행 × 170열입니다. `class` 한 열이 빠졌습니다.

정답 `y`를 만듭니다. APS 고장(`pos`)이면 1, 아니면 0입니다.

```python
y = (train["class"] == "pos").astype(int)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


`y`의 합은 1의 개수, 곧 APS 고장 건수입니다.

```python
y.sum()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `1000`이면 맞습니다. 위에서 센 `pos` 1,000건과 같습니다.

---
## 2. 두 오류를 세는 함수

두 오류와 비용은 다음과 같습니다. 맞힌 판정의 비용은 0입니다.

| | 정답 APS(1) | 정답 다른 고장(0) |
|---|---|---|
| **APS로 판정(1)** | 맞힘 | **오경보 `fp`** — 비용 10 |
| **다른 고장으로 판정(0)** | **미탐지 `fn`** — 비용 500 | 맞힘 |

정답 `y_true`와 판정 `y_pred`를 받아 오경보와 미탐지 수를 돌려주는 함수 `errors`를 만듭니다. 이번 주 내내 씁니다. `&`는 두 조건을 모두 만족하는 행, `.sum()`은 그 개수, `return fp, fn`은 두 값을 함께 돌려줍니다.

```python
def errors(y_true, y_pred):
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return fp, fn
```

In [ ]:
# ✍️ 위 코드를 입력하세요


---
## 3. 학습용과 검증용으로 나눈다

학습 파일 60,000건을 학습용 48,000건과 검증용 12,000건으로 나눕니다. 모델은 학습용으로 만들고, 판정 기준은 검증용으로 고릅니다. 테스트 파일은 고른 기준을 마지막에 평가하는 데만 씁니다.

`train_test_split`에서 `test_size`는 떼어 낼 건수, `stratify=y`는 두 쪽의 APS 비율을 같게, `random_state=0`은 매번 같은 결과가 나오게 합니다.

```python
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=12000, stratify=y, random_state=0)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


학습용 입력의 크기를 봅니다.

```python
X_train.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


검증용 입력의 크기를 봅니다.

```python
X_val.shape
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 학습용 48,000건, 검증용 12,000건입니다. 열은 둘 다 170개입니다.

학습용에 APS가 몇 건 들어갔는지 셉니다.

```python
y_train.sum()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


검증용의 APS 건수도 셉니다.

```python
y_val.sum()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 학습용 `800`, 검증용 `200`이면 맞습니다. 두 쪽 모두 60건 중 1건꼴로, `stratify`가 비율을 맞췄습니다.

모델 없이 모두 "APS 아님"(0)으로 판정하면 검증용에서 두 오류가 몇 건인지 셉니다. 정답과 길이가 같은 0 판정은 `y_val * 0`입니다.

```python
errors(y_val, y_val * 0)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 `(0, 200)` — 오경보는 없고 APS 200건을 모두 놓칩니다. 비용은 200 × 500 = **100,000**입니다.

반대로 모두 "APS"(1)로 판정하면 어떤지 셉니다. 1 판정은 `y_val * 0 + 1`입니다.

```python
errors(y_val, y_val * 0 + 1)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 `(11800, 0)` — 미탐지는 없고 다른 고장 11,800건이 모두 오경보입니다. 비용은 11,800 × 10 = **118,000**입니다. 모델로 만든 판정은 이 두 값보다 훨씬 낮아야 쓸모가 있습니다.

---
## 4. 모델로 판정 점수를 만든다

### 빈 값 채우기

입력에는 빈 값이 있어 모델에 넣기 전에 채웁니다. 학습용의 열별 중앙값으로 학습용과 검증용을 채웁니다. 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 학습용 중앙값으로 빈 값 채우기
train_median = X_train.median()
X_train = X_train.fillna(train_median)
X_val = X_val.fillna(train_median)

### 모델 만들기

**LightGBM** 모델을 만듭니다. 결정 트리를 여러 개 순서대로 더해 가며, 앞의 트리가 틀린 부분을 다음 트리가 고치는 모델입니다. 트리 수·학습률 같은 설정은 고정해 두었으므로 실행만 합니다(설정 뜻은 배경지식 6절). 이번 주에 고를 것은 모델 설정이 아니라 판정 기준입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 고정 설정의 LightGBM 모델
model = LGBMClassifier(
    num_leaves=15, learning_rate=0.03, n_estimators=800,
    min_child_samples=20, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, verbose=-1, random_state=0,
    n_jobs=1, deterministic=True, force_col_wise=True)

### 학습하기

`model.fit(입력, 정답)`으로 학습용 자료를 학습합니다. 30초~1분 걸립니다.

```python
model.fit(X_train, y_train)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


### 검증 기록에 점수 매기기

`model.predict_proba(입력)`은 기록마다 두 클래스의 확률을 냅니다. 결과를 `proba`에 담습니다.

```python
proba = model.predict_proba(X_val)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


앞의 세 행을 봅니다.

```python
proba[:3]
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 한 행에 숫자가 두 개이고, 더하면 1입니다. 첫째 열은 다른 고장(0)일 확률, 둘째 열은 **APS(1)일 확률**입니다. 첫 세 기록은 모두 APS일 확률이 0에 가깝습니다.

둘째 열(`[:, 1]`)만 꺼내 검증 점수 `proba_val`로 둡니다. `index=X_val.index`는 정답 `y_val`과 행을 맞춥니다.

```python
proba_val = pd.Series(proba[:, 1], index=X_val.index)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


점수의 분포를 요약합니다.

```python
proba_val.describe()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `count 12000`, `50%`가 `1.49e-05` 근처면 맞습니다.

💬 점수의 중앙값은 **0.0000149**로, 대부분의 기록이 0에 아주 가깝고 소수만 높습니다(최대 0.99994). 이 점수를 판정으로 바꾸는 기준을 5절에서 정합니다.

---
## 5. 비용으로 기준 정하기

점수가 **기준** 이상이면 APS로 판정합니다. 흔히 0.5를 쓰지만, 이 문제는 두 오류의 비용이 달라 비용으로 기준을 정합니다. APS일 확률이 p인 기록을 APS로 판정하면 평균 비용은 (1 − p) × 10, APS 아님으로 판정하면 p × 500입니다. 점수가 0.03인 기록으로 두 값을 비교합니다.

```python
p = 0.03
(1 - p) * 10, p * 500
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 `(9.7, 15.0)` — APS로 판정하는 쪽이 쌉니다. 두 값이 같아지는 p가 기준입니다.

```python
10 / (10 + 500)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `0.0196...`이면 맞습니다.

💬 **APS일 확률이 0.0196(약 2%) 이상이면 APS로 판정**하는 것이 평균 비용이 낮습니다(비용식 기준). 모델 점수가 실제 확률과 꼭 맞지는 않으므로 7절에서 검증 자료로 실제 비용이 가장 낮은 기준을 찾습니다.

---
## 6. 기준 하나로 비용 세기

비용식 기준 0.0196으로 검증 기록을 판정합니다. 점수가 기준 이상이면 1, 아니면 0입니다.

```python
y_pred = (proba_val >= 10 / (10 + 500)).astype(int)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


APS로 판정한 기록이 몇 건인지 봅니다.

```python
y_pred.sum()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 검증 12,000건 중 359건을 APS로 판정했습니다. 실제 APS는 200건입니다.

두 오류를 셉니다.

```python
fp, fn = errors(y_val, y_pred)
fp, fn
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 오경보 167건, 미탐지 8건입니다.

두 오류에 단가를 곱해 비용을 구합니다.

```python
10 * fp + 500 * fn
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `5670`이면 맞습니다. 3절의 두 극단(100,000·118,000)보다 훨씬 낮습니다.

---
## 7. 가장 싼 기준 찾기

6절의 계산을 여러 기준에 반복해 검증 비용이 가장 낮은 기준을 찾습니다. 기준 후보는 검증 점수의 서로 다른 값 전부이고, 앞에 0(전원 "APS"), 뒤에 무한대(전원 "APS 아님")를 붙입니다. 판정은 기준이 관측된 점수를 지날 때만 바뀌므로 이것으로 가능한 판정을 모두 비교할 수 있습니다.

```python
thresholds = [0.0] + sorted(proba_val.unique()) + [float("inf")]
len(thresholds)
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 후보는 11,720개입니다.

후보마다 판정 → 두 오류를 `rows`에 모읍니다. 한 행은 [기준, 오경보, 미탐지, APS 판정 수]입니다. 15초쯤 걸립니다.

```python
rows = []
for threshold in thresholds:
    y_pred = (proba_val >= threshold).astype(int)
    fp, fn = errors(y_val, y_pred)
    rows.append([threshold, fp, fn, y_pred.sum()])
```

In [ ]:
# ✍️ 위 코드를 입력하세요


표 `curve`로 만들고 앞부분을 봅니다.

```python
curve = pd.DataFrame(rows, columns=["threshold", "fp", "fn", "positive"])
curve.head()
```

In [ ]:
# ✍️ 위 코드를 입력하세요


💬 첫 행(기준 0)은 전원 "APS" 판정이라 오경보 11,800건입니다. 기준이 올라갈수록 오경보가 줄어듭니다.

후보마다 비용 열을 붙입니다.

```python
curve["cost"] = 10 * curve["fp"] + 500 * curve["fn"]
```

In [ ]:
# ✍️ 위 코드를 입력하세요


비용이 가장 낮은 행을 고릅니다. `.loc[조건]`으로 최소 비용인 행만 남기고, `.iloc[-1]`로 그중 마지막 행을 고릅니다. 비용이 같은 기준이 여럿이면 가장 큰 기준(마지막 행)을 쓰기로 합니다.

```python
best = curve.loc[curve["cost"] == curve["cost"].min()].iloc[-1]
best
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `threshold 0.014902`, `fp 186`, `fn 6`, `positive 380`, `cost 4860`이면 맞습니다.

💬 **결과 해설**
- 검증에서 가장 싼 기준은 약 **0.0149**로, 0.5보다 훨씬 낮고 비용식 기준(0.0196)과 가깝습니다.
- 오경보 186건(1,860) + 미탐지 6건(3,000) = **4,860**으로, 비용식 기준(5,670)보다 810 쌉니다.
- 기준을 고른 자료의 비용입니다. 처음 보는 기록의 비용은 8절에서 테스트로 확인합니다.

모든 후보의 비용을 그림으로 봅니다(실행만). 가로축은 로그 눈금, 빨간 선은 고른 기준, 회색 점선은 비용식 기준입니다.

In [ ]:
# ▶ 그대로 실행하세요 — 검증 비용 곡선 (0과 무한대는 로그 축에 그릴 수 없어 뺍니다)
plot_curve = curve.loc[(curve["threshold"] > 0) & (curve["threshold"] < float("inf"))]
plt.figure(figsize=(8, 4))
plt.plot(plot_curve["threshold"], plot_curve["cost"])
plt.xscale("log")
plt.axvline(best["threshold"], color="red", label="Selected on validation")
plt.axvline(10 / 510, color="gray", linestyle="--", label="Cost formula")
plt.xlabel("Threshold")
plt.ylabel("Validation cost")
plt.legend()
plt.show()

💬 기준을 낮추면 미탐지가 줄고 오경보가 늘어, 아주 낮으면 오경보 비용이, 높으면 미탐지 비용이 커집니다. 그 사이의 가장 낮은 곳에서 기준을 골랐습니다.

---
## 8. 테스트에서 0.5와 비교

테스트 파일에 학습용 중앙값과 같은 모델을 적용해 정답 `y_test`와 점수 `proba_test`를 만듭니다. 4절과 같은 방법이므로 실행만 합니다.

In [ ]:
# ▶ 그대로 실행하세요 — 테스트 입력·정답·점수 (모델은 다시 학습하지 않습니다)
X_test = test.drop(columns=["class"]).fillna(train_median)
y_test = (test["class"] == "pos").astype(int)
proba_test = pd.Series(model.predict_proba(X_test)[:, 1], index=X_test.index)
y_test.sum()

💬 테스트 16,000건 중 APS는 375건입니다.

기본값 0.5로 판정한 테스트 비용을 구합니다.

```python
fp, fn = errors(y_test, (proba_test >= 0.5).astype(int))
cost_default = 10 * fp + 500 * fn
fp, fn, cost_default
```

In [ ]:
# ✍️ 위 코드를 입력하세요


검증에서 고른 기준으로 판정한 테스트 비용을 구합니다.

```python
fp, fn = errors(y_test, (proba_test >= best["threshold"]).astype(int))
cost_best = 10 * fp + 500 * fn
fp, fn, cost_best
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ 0.5는 `(23, 82, 41230)`, 고른 기준은 `(234, 24, 14340)`이면 맞습니다.

고른 기준의 비용이 기본값의 몇 배인지 계산합니다.

```python
cost_best / cost_default
```

In [ ]:
# ✍️ 위 코드를 입력하세요


✅ `0.3478...`이면 맞습니다.

💬 **결과 해설**
- 미탐지가 82건에서 24건으로 줄고 오경보가 23건에서 234건으로 늘었습니다. 미탐지 58건 × 500이 오경보 211건 × 10보다 훨씬 커서 비용이 **41,230 → 14,340, 약 0.35배**가 됐습니다.
- 검증 비용(4,860)과 테스트 비용(14,340)은 건수(12,000·16,000)와 APS 수(200·375)부터 다릅니다. 이 결과를 본 뒤 기준을 바꾸지 않습니다.

---
## 9. 결정

> **오경보 10, 미탐지 500의 비용에서는 기본값 0.5 대신 검증에서 고른 기준 0.0149 이상을 APS로 판정한다.** 테스트 비용이 41,230에서 14,340으로(약 0.35배) 줄었다.

- 0.5는 두 오류를 같은 가격으로 볼 때의 기준입니다. 미탐지가 50배 비싸면 기준을 크게 낮춰야 합니다.
- 0.0149는 이 모델·자료·비용표에 맞춘 값입니다. 비용이 바뀌면 `curve`에서 비용 열만 다시 계산해 고릅니다(랩).
- 이 오류 수는 이미 고장 난 기록을 APS와 다른 고장으로 가려낸 결과입니다. 줄어든 미탐지를 예방한 고장 수로 읽지 않습니다.

---
## 10. 실습 과제 (10분)

### 📝 과제 1 · 정확도가 가장 높은 기준을 고르면?

7절의 `curve`에 검증 정확도 열 `accuracy`를 붙이고, 정확도가 가장 높은 행을 고르세요. 정확도는 1에서 (오경보 + 미탐지) / 검증 건수를 뺀 값입니다. 그 행의 기준·비용을 고른 기준(4,860)과 비교합니다.

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 7절에서 cost 열을 붙인 것처럼 accuracy 열을 만들고, 7절의 best 한 줄에서 cost와 min을 accuracy와 max로 바꿉니다.


✏️ **나의 답:** (여기를 두 번 눌러 적습니다)

### 📝 과제 2 · 생각해 보기

- 기준을 고른 검증 자료로 그 기준의 비용을 다시 보고하면 무엇이 문제일까요? 테스트 파일을 마지막까지 쓰지 않은 이유와 함께 적으세요.
- 8절의 결과만으로 "고장을 미리 막았다"고 말할 수 없는 이유는 무엇인가요?

✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 두 오류 | 오경보(비용 10), 미탐지(비용 500). `errors(y_true, y_pred)` → `(fp, fn)` |
| 자료의 역할 | 학습용 48,000건(모델) · 검증용 12,000건(기준 고르기) · 테스트 16,000건(마지막 평가) |
| 판정 점수 | `predict_proba`의 둘째 열 = APS일 확률. 대부분 0에 가깝다 |
| 비용식 기준 | (1 − p) × 10 = p × 500 → p = 10 ÷ 510 ≈ 0.0196 |
| 검증에서 고른 기준 | 모든 후보의 비용을 세어 가장 싼 0.0149(비용 4,860) |
| 테스트 | 0.5는 41,230, 고른 기준은 14,340 → 약 0.35배 |

### 이번 주에 할 일

| 활동 | 자료 | 비고 |
|---|---|---|
| 배경지식 | [판정과 비용](https://balab-pknu.github.io/bizanalytics/theory/week05.html) | 두 오류, 비용에서 기준 도출, 자료의 역할 |
| 교재 | 이 노트북 | |
| 랩 | `week05_lab.ipynb` — 같은 APS 데이터 | 오류의 비용이 바뀌면 판정 기준은 어떻게 달라질까? 약 60분 |